# Pruning Some Layer

## Install TensorFlow Model Optimization Toolkit
* 설치 완료 후 반드시 Runtime 재시작!
    * '런타임' > '세션 다시 시작' 메뉴 선택

기존 학습 모델
     ↓
Dense Layer에 Pruning 적용
     ↓
Fine-tuning
     ↓
weight 중 작은 값들을 0으로 만듦
     ↓
Pruning 전/후 정확도 비교
     ↓
Pruning 정보 제거(strip_pruning)
     ↓
LiteRT(TFLite) 변환
     ↓
속도 비교
     ↓
압축 파일 크기 비교

In [ ]:
!pip install tensorflow-model-optimization

## Mount Google driver

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('g-drive mounted.')
    colab=True
except:
    print('local drive.')
    colab =False

Mounted at /content/drive
g-drive mounted.


In [ ]:
if colab :
  save_dir = '/content/drive/MyDrive/files/save/'
else :
  save_dir = '../files/save/'

## Import Module

In [ ]:
import tensorflow as tf
import numpy as np

import tensorflow_model_optimization as tfmot
from tensorflow_model_optimization.python.core.keras.compat import keras

print(tf.__version__)
print(np.__version__)

2.19.0
1.26.4


## Load Dataset

In [ ]:
(train_images, train_labels), (test_images, test_labels) = keras.datasets.mnist.load_data()

train_images = (train_images / 255.0).astype(np.float32)
test_images = (test_images / 255.0).astype(np.float32)

11490434/11490434 [==============================] - 0s 0us/step


In [ ]:
train_images = np.expand_dims(train_images, axis=-1)
test_images = np.expand_dims(test_images, axis=-1)

## Load CNN Model for MNIST

In [ ]:
model = keras.models.load_model(save_dir + 'baseline_model.h5') # 이미 학습이 끝난 모델 불러옴
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 26, 26, 32)        320       
                                                                 
 max_pooling2d (MaxPooling2  (None, 13, 13, 32)        0         
 D)                                                              
                                                                 
 conv2d_1 (Conv2D)           (None, 11, 11, 16)        4624      
                                                                 
 max_pooling2d_1 (MaxPoolin  (None, 5, 5, 16)          0         
 g2D)                                                            
                                                                 
 flatten (Flatten)           (None, 400)               0         
                                                                 
 dense (Dense)               (None, 128)               5

In [ ]:
_, baseline_model_accuracy = model.evaluate(
    test_images, test_labels, verbose=0)

print('Baseline test accuracy:', baseline_model_accuracy) # pruning전 기존 모델의 정확도 출력

Baseline test accuracy: 0.9904000163078308


## Check weight before pruning (dense layer)

In [ ]:
model.layers[5].get_weights() # 모델의 6번째 layer를 가져온다.( 6번째 레이어가 dense라서 / 단순 확인용,없어도 된다)

[array([[ 0.11715356,  0.13895866, -0.0544957 , ..., -0.3218954 ,
         -0.09052682,  0.10737804],
        [ 0.25028417,  0.25553143, -0.06422829, ..., -0.22241999,
         -0.23787414,  0.12104833],
        [-0.0990324 ,  0.24197778,  0.06178116, ..., -0.18422675,
         -0.03463924,  0.01239588],
        ...,
        [ 0.18176717, -0.13811027,  0.04270137, ...,  0.20849746,
         -0.04789718,  0.03234704],
        [-0.02691392, -0.08045828, -0.17388692, ...,  0.0877137 ,
         -0.01064873, -0.03722059],
        [-0.14850925,  0.09677936, -0.12765685, ..., -0.09072027,
          0.04380804,  0.06439313]], dtype=float32),
 array([ 0.04535642,  0.0531471 ,  0.02759967, -0.01055882, -0.07299803,
        -0.01139281,  0.01678417,  0.0383155 ,  0.01824879,  0.04456723,
        -0.05497947, -0.04973326, -0.01158212,  0.03233543, -0.00944897,
        -0.0416717 ,  0.01646345,  0.02749427,  0.03071346,  0.01257269,
        -0.0201099 ,  0.06118304,  0.01892779,  0.01920945, -0.020

## Fine-tune pre-trained model with pruning (Some Layer)
* 적용 Layer : 마지막 2개의 Dense Layer에 적용
    * Scheduler : tfmot.sparsity.keras.PolynomialDecay
        * Initial sparsity : 50% (50% zeros in weights)
        * End sparsigy : 80% sparsity.

In [ ]:
batch_size = 128 # 한번에 128개
epochs = 2 # 전체데이터 2번 학습
validation_split = 0.1 # 10프로를 validation으로 사용

num_images = train_images.shape[0] * (1 - validation_split) # 학습할 이미지 갯수
end_step = np.ceil(num_images / batch_size).astype(np.int32) * epochs # 학습 step수(np.ceil() : 올림)

# pruning 설정
pruning_params = {
      'pruning_schedule': tfmot.sparsity.keras.PolynomialDecay( # pruning비율을 학습 과정에서 점점 증가시키는 scheduler
            initial_sparsity=0.50, # w 중 약 50% 를 0으로 만들기
            final_sparsity=0.80, # w 의 80%를 0으로 만드는걸 목표 / 50% -> 80%
            begin_step=0, # pruning 0번째 step부터
            end_step=end_step) # 마지막 step까지
}

In [ ]:
def apply_pruning_to_dense(layer): # layer 하나씩 받아서 함수 돌리자
  if isinstance(layer, keras.layers.Dense): # 지금 레이어가 dense 레이어라면
    # Dense layer는 pruning 적용 layer return
    return tfmot.sparsity.keras.prune_low_magnitude(layer, **pruning_params) # pruning을 적용해서 리턴하자
  return layer # dense가 아니면 그대로 리턴

model_for_pruning = keras.models.clone_model( # 기존 모델을 복제해서 새로운 모델 만들자
    model,
    clone_function=apply_pruning_to_dense, # 각 레이어를 검사하면서 dense layer만 pruning적용
)

model_for_pruning.compile(optimizer='adam', # 모델 컴파일 하자 
              loss=keras.losses.SparseCategoricalCrossentropy(),
              metrics=['accuracy'])

model_for_pruning.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 26, 26, 32)        320       
                                                                 
 max_pooling2d (MaxPooling2  (None, 13, 13, 32)        0         
 D)                                                              
                                                                 
 conv2d_1 (Conv2D)           (None, 11, 11, 16)        4624      
                                                                 
 max_pooling2d_1 (MaxPoolin  (None, 5, 5, 16)          0         
 g2D)                                                            
                                                                 
 flatten (Flatten)           (None, 400)               0         
                                                                 
 prune_low_magnitude_dense   (None, 128)               1

In [ ]:
callbacks = [
  tfmot.sparsity.keras.UpdatePruningStep(), # pruning scheduler 가 현재 학습 step을 알수있게 계속 업데이트
# 현재 몇 번째 학습 step인가?
#         ↓
# Pruning Scheduler에게 알려줌
#         ↓
# 현재 pruning 비율 결정
]

hist_pruning = model_for_pruning.fit(train_images, train_labels, # pruning이적용 모델 학습하기
                  batch_size=batch_size, epochs=epochs, validation_split=validation_split,
                  callbacks=callbacks)

Epoch 1/2
422/422 [==============================] - 12s 15ms/step - loss: 0.0119 - accuracy: 0.9961 - val_loss: 0.0409 - val_accuracy: 0.9883
Epoch 2/2
422/422 [==============================] - 2s 6ms/step - loss: 0.0131 - accuracy: 0.9959 - val_loss: 0.0358 - val_accuracy: 0.9912


In [ ]:
_, model_for_pruning_accuracy = model_for_pruning.evaluate( # 정확도 측정
   test_images, test_labels, verbose=0)

print('Baseline test accuracy:', baseline_model_accuracy)
print('Pruned test accuracy:', model_for_pruning_accuracy)

Baseline test accuracy: 0.9904000163078308
Pruned test accuracy: 0.9894000291824341


In [ ]:
model_for_export = tfmot.sparsity.keras.strip_pruning(model_for_pruning) 
# pruning 관리를 위해 pruning관리 정보들이 더 있다.
# 이제 학습이 끝났으니 이런 정보들은 필요없고 w가 0이된 결과만 갖고싶다.
# 그런 불필요한 정보들을 떼어내는 과정

model_for_export.layers[5].get_weights() # dense 레이어 눈으로 보기

[array([[ 0.        ,  0.        ,  0.        , ..., -0.34895945,
          0.        , -0.        ],
        [ 0.24533811,  0.29001987,  0.        , ..., -0.21523741,
         -0.27781937, -0.        ],
        [-0.        ,  0.26371992, -0.        , ..., -0.18422315,
         -0.        ,  0.        ],
        ...,
        [ 0.17882732,  0.        , -0.        , ...,  0.2114186 ,
         -0.        , -0.        ],
        [ 0.        ,  0.        , -0.20481661, ..., -0.        ,
          0.        , -0.        ],
        [-0.        ,  0.        , -0.        , ..., -0.        ,
          0.        ,  0.        ]], dtype=float32),
 array([ 0.07903216,  0.06276814,  0.06463092, -0.00849664, -0.03940045,
        -0.01139281,  0.01990275,  0.04010355,  0.00837971,  0.07784203,
        -0.01923754, -0.05400397,  0.0050429 ,  0.02919529,  0.00943381,
        -0.02351368, -0.00055652,  0.00664447,  0.01966929,  0.02933761,
         0.01662352,  0.09002434,  0.00986183,  0.01262911, -0.020

In [ ]:
total, non_zero = 0, 0 # total: 전체 파라미터 갯수, non_zero: 0이 아닌 파라미터 갯수

for l in model_for_export.layers: # 모델의 모든 레이어 동안 반복해라
    weights = l.get_weights() # 현재 레이어의 w, bias 가져와
    for i, w in enumerate(weights):
        if type(w) == np.ndarray: # w가 numpy배열이라면
            rate = (w.size - np.count_nonzero(w))/w.size # w =0 인 비율
            print("layer [{}] / weight [{}] : rate = {}".format(l.name, i, rate))
            total += w.size # 현재 파라미터 갯수 + 현재 w 개수
            non_zero += np.count_nonzero(w) # + 0이 아닌 파라미터 갯수 

print( "=" * 40)
print( "Total parameter : {}".format(total))
print( "Non-zero parameter : {}".format(non_zero))
print( "Rate of pruned parmeter : {}".format((total-non_zero)/ total))

layer [conv2d] / weight [0] : rate = 0.0
layer [conv2d] / weight [1] : rate = 0.0
layer [conv2d_1] / weight [0] : rate = 0.0
layer [conv2d_1] / weight [1] : rate = 0.0
layer [dense] / weight [0] : rate = 0.7999609375
layer [dense] / weight [1] : rate = 0.0
layer [dense_1] / weight [0] : rate = 0.8
layer [dense_1] / weight [1] : rate = 0.0
Total parameter : 57562
Non-zero parameter : 15580
Rate of pruned parmeter : 0.729335325388277


In [ ]:
model_for_export.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 26, 26, 32)        320       
                                                                 
 max_pooling2d (MaxPooling2  (None, 13, 13, 32)        0         
 D)                                                              
                                                                 
 conv2d_1 (Conv2D)           (None, 11, 11, 16)        4624      
                                                                 
 max_pooling2d_1 (MaxPoolin  (None, 5, 5, 16)          0         
 g2D)                                                            
                                                                 
 flatten (Flatten)           (None, 400)               0         
                                                                 
 dense (Dense)               (None, 128)               5

## LiteRT 모델로 변환 (Pruning some layer)

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model_for_export) # keras -> LiteRT 위한 컨버터
tflite_model = converter.convert() # keras -> LiteRT 변환

In [ ]:
tflite_pruning_some_layer_file = save_dir + 'mnist_pruning_some_layer.tflite'
open(tflite_pruning_some_layer_file, 'wb').write(tflite_model)

233596

## 추론 속도 측정

* benchmark_model 설치

In [ ]:
!wget https://storage.googleapis.com/tensorflow-nightly-public/prod/tensorflow/release/lite/tools/nightly/latest/linux_x86-64_benchmark_model
!chmod +x linux_x86-64_benchmark_model

--2025-12-21 14:39:22--  https://storage.googleapis.com/tensorflow-nightly-public/prod/tensorflow/release/lite/tools/nightly/latest/linux_x86-64_benchmark_model
Resolving storage.googleapis.com (storage.googleapis.com)... 142.251.2.207, 74.125.137.207, 142.250.141.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|142.251.2.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 6685264 (6.4M) [application/octet-stream]
Saving to: ‘linux_x86-64_benchmark_model’

linux_x86-64_benchm 100%[===================>]   6.38M  --.-KB/s    in 0.03s   

2025-12-21 14:39:22 (223 MB/s) - ‘linux_x86-64_benchmark_model’ saved [6685264/6685264]



* Baseline model

In [ ]:
tflite_baseline_model_file = save_dir + 'mnist_baseline_model.tflite'
cmd = f'./linux_x86-64_benchmark_model --graph={tflite_baseline_model_file}'
print(cmd)
!{cmd}

./linux_x86-64_benchmark_model --graph=/content/drive/MyDrive/files/save/mnist_baseline_model.tflite
INFO: STARTING!
INFO: Log parameter values verbosely: [0]
INFO: Graph: [/content/drive/MyDrive/files/save/mnist_baseline_model.tflite]
INFO: Signature to run: []
INFO: Loaded model /content/drive/MyDrive/files/save/mnist_baseline_model.tflite
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
INFO: The input model file size (MB): 0.233596
INFO: Initialized session in 379.824ms.
INFO: Running benchmark for at least 1 iterations and at least 0.5 seconds but terminate if exceeding 150 seconds.
INFO: count=13936 first=71 curr=32 min=31 max=1162 avg=35.6165 std=14 p5=32 median=35 p95=46

INFO: Running benchmark for at least 50 iterations and at least 1 seconds but terminate if exceeding 150 seconds.
INFO: count=28042 first=52 curr=40 min=26 max=225 avg=35.4061 std=6 p5=31 median=32 p95=50

INFO: Inference timings in us: Init: 379824, First inference: 71, Warmup (avg): 35.6165, Inference

* Pruning some layer

In [ ]:
cmd = f'./linux_x86-64_benchmark_model --graph={tflite_pruning_some_layer_file}'
print(cmd)
!{cmd}

./linux_x86-64_benchmark_model --graph=/content/drive/MyDrive/files/save/mnist_pruning_some_layer.tflite
INFO: STARTING!
INFO: Log parameter values verbosely: [0]
INFO: Graph: [/content/drive/MyDrive/files/save/mnist_pruning_some_layer.tflite]
INFO: Signature to run: []
INFO: Loaded model /content/drive/MyDrive/files/save/mnist_pruning_some_layer.tflite
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
INFO: The input model file size (MB): 0.233596
INFO: Initialized session in 4.715ms.
INFO: Running benchmark for at least 1 iterations and at least 0.5 seconds but terminate if exceeding 150 seconds.
INFO: count=13872 first=78 curr=32 min=26 max=652 avg=35.7804 std=10 p5=27 median=32 p95=53

INFO: Running benchmark for at least 50 iterations and at least 1 seconds but terminate if exceeding 150 seconds.
INFO: count=27600 first=50 curr=32 min=26 max=1133 avg=35.9743 std=14 p5=31 median=32 p95=53

INFO: Inference timings in us: Init: 4715, First inference: 78, Warmup (avg): 35.7804, 

## 모델 압축 테스트

* 압축 함수 정의

In [ ]:
import tempfile
import os
import zipfile

def get_zipped_model_size(model):

  _, model_file = tempfile.mkstemp('.h5')
  model.save(model_file)

  _, zipped_file = tempfile.mkstemp('.zip')
  with zipfile.ZipFile(zipped_file, 'w', compression=zipfile.ZIP_DEFLATED) as f:
    f.write(model_file)

  return os.path.getsize(zipped_file)

* 압축된 파일 크기 비교

In [ ]:
size_zipped_baseline_model = get_zipped_model_size(model)
size_zipped_pruning_model = get_zipped_model_size(model_for_export)

print("Size of zipped baseline model file : {}".format(size_zipped_baseline_model))
print("Size of zipped pruning model file : {}".format(size_zipped_pruning_model))
print("ratio : {}".format(size_zipped_baseline_model/size_zipped_pruning_model))

/usr/local/lib/python3.12/dist-packages/tf_keras/src/engine/training.py:3098: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native TF-Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


Size of zipped baseline model file : 502096
Size of zipped pruning model file : 83628
ratio : 6.003922131343569
